# Proyecto Integrador: Métodos Numéricos — Entrega Final (Fase 3: Consolidación, Validación y Análisis)

**Título:** Cubatura Numérica Adaptativa sobre Datos Dispersos: Estimación de la Carga Atmosférica de Material Particulado ($PM_{10}$) en Bogotá  
**Artículo Base:** *Numerical cubature on scattered data by adaptive interpolation* (Cavoretto, De Rossi & Erb, 2024)  
**Integrantes:** Santiago Sandoval Cortés, Miguel David Moreno Lozano, Luis Alejandro Camacho Benítez  
**Profesor:** Edwin Fernando Muñoz Chipatecua | **Universidad:** Universidad Externado de Colombia  
**Semilla Global Fija:** `SEED = 42` (100% reproducible de principio a fin)

---
### Novedades Metodológicas Consolidadas en la Entrega Final:
1. **Cálculo del Orden Empírico de Convergencia (EOC, $p_k = \log_2(E_{k-1}/E_k)$)** sobre la función analítica de Franke y verificación cuantitativa del estancamiento (*Error Stalling*).
2. **Validación Numérica Formal contra Librería de Referencia (`matplotlib.tri.LinearTriInterpolator`)**, comprobando coincidencia en el orden del épsilon de máquina ($2.13 \times 10^{-13}$).
3. **Justificación Teórica del Grado de Precisión Algebraico:** Demostración de por qué los polinomios cuadráticos y la interpolación lineal baricéntrica no activan refinamiento adaptativo (grado de precisión 2 de la regla de Gauss de 3 puntos).
4. **Análisis de Sensibilidad de Hiperparámetros de Shepard ($k_{\mathrm{neighbors}}$)** sobre la red urbana de Bogotá.
5. **Cero Cajas Negras:** 100% de los algoritmos (Householder QR, PA=LU, Shepard, Delaunay baricéntrico, Dunavant 7 pts y Gauss 3 pts) implementados desde cero.

## 1. Configuración, Semilla y Librerías Base


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# Fijación de semilla global para reproducibilidad matemática estricta
SEED = 42
np.random.seed(SEED)

# Estilo gráfico coherente con el informe
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

print("Librerías importadas y semilla fijada en SEED =", SEED)


## 2. Álgebra Lineal Numérica: Solvers Directos desde Cero
Se implementan:
* `forward_substitution` y `backward_substitution`
* `lu_partial_pivot`: Factorización $PA = LU$ con pivoteo parcial por filas.
* `householder_qr`: Factorización $A = QR$ reducida mediante reflexiones de Householder, evitando cancelación catastrófica (IEEE-754).
* `solve_least_squares_normal_lu`: Resuelve $(A^T A)\mathbf{c} = A^T\mathbf{z}$ vía LU.
* `solve_least_squares_qr`: Resuelve $R\mathbf{c} = Q^T\mathbf{z}$ vía QR sin formar $A^T A$.


In [ ]:
"""
Módulo: linear_solvers.py
Implementación desde cero de métodos directos de Álgebra Lineal Numérica:
1. Sustitución progresiva (Forward Substitution) y regresiva (Backward Substitution).
2. Factorización LU con Pivoteo Parcial (PA = LU) aplicada a Ecuaciones Normales.
3. Factorización QR mediante Reflexiones de Householder (A = QR) para Mínimos Cuadrados.
4. Diagnóstico de número de condición L2 y pérdida de ortogonalidad en precisión IEEE-754.

Autores: Santiago Sandoval Cortés, Miguel David Moreno Lozano, Luis Alejandro Camacho Benítez
"""



def forward_substitution(L: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Resuelve el sistema triangular inferior L * y = b desde cero.
    """
    n = L.shape[0]
    y = np.zeros(n, dtype=np.float64)
    for i in range(n):
        soma = 0.0
        for j in range(i):
            soma += L[i, j] * y[j]
        if abs(L[i, i]) < 1e-15:
            raise ValueError(f"Pivote nulo o casi singular en L[{i},{i}] = {L[i,i]}")
        y[i] = (b[i] - soma) / L[i, i]
    return y


def backward_substitution(U: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Resuelve el sistema triangular superior U * c = y desde cero.
    """
    n = U.shape[0]
    c = np.zeros(n, dtype=np.float64)
    for i in range(n - 1, -1, -1):
        soma = 0.0
        for j in range(i + 1, n):
            soma += U[i, j] * c[j]
        if abs(U[i, i]) < 1e-15:
            raise ValueError(f"Pivote nulo o casi singular en U[{i},{i}] = {U[i,i]}")
        c[i] = (y[i] - soma) / U[i, i]
    return c


def lu_partial_pivot(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calcula la factorización PA = LU con pivoteo parcial por filas desde cero.
    Retorna (P, L, U) tales que P @ A = L @ U.
    """
    n = A.shape[0]
    U = A.astype(np.float64).copy()
    L = np.eye(n, dtype=np.float64)
    P = np.eye(n, dtype=np.float64)

    for k in range(n - 1):
        # Selección del pivote máximo en la columna k desde la fila k hasta n-1
        pivot_row = k + int(np.argmax(np.abs(U[k:, k])))
        if abs(U[pivot_row, k]) < 1e-15:
            raise ValueError(f"Matriz numéricamente singular en paso k={k}.")

        # Intercambio de filas en U, P y subdiagonal de L
        if pivot_row != k:
            U[[k, pivot_row], k:] = U[[pivot_row, k], k:]
            P[[k, pivot_row], :] = P[[pivot_row, k], :]
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]

        # Eliminación gaussiana hacia adelante
        for i in range(k + 1, n):
            factor = U[i, k] / U[k, k]
            L[i, k] = factor
            U[i, k:] = U[i, k:] - factor * U[k, k:]
            U[i, k] = 0.0

    return P, L, U


def householder_qr(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Calcula la Factorización QR reducida (A = Q @ R) mediante Reflexiones de Householder
    implementadas desde cero. Evita la cancelación catastrófica seleccionando el signo
    del reflector de Householder opuesto a la primera componente.
    
    Parámetros:
        A: Matriz de diseño de dimensión (m, n) con m >= n.
    Retorna:
        Q_red: Matriz con columnas ortonormales de dimensión (m, n).
        R_red: Matriz triangular superior de dimensión (n, n).
    """
    m, n = A.shape
    R = A.astype(np.float64).copy()
    Q_full = np.eye(m, dtype=np.float64)

    for k in range(n):
        x = R[k:m, k].copy()
        norm_x = np.sqrt(np.dot(x, x))
        if norm_x < 1e-15:
            continue

        # Elección de signo para evitar cancelación numérica (IEEE-754)
        sign_x0 = 1.0 if x[0] >= 0.0 else -1.0
        v = x.copy()
        v[0] = v[0] + sign_x0 * norm_x
        norm_v_sq = np.dot(v, v)

        if norm_v_sq < 1e-30:
            continue

        # Aplicar reflector H_k = I - 2 * (v v^T) / (v^T v) a la submatriz R[k:m, k:n]
        for j in range(k, n):
            dot_vj = np.dot(v, R[k:m, j])
            R[k:m, j] -= (2.0 * dot_vj / norm_v_sq) * v

        # Acumular Q_full = Q_full @ H_k
        for i in range(m):
            dot_iv = np.dot(Q_full[i, k:m], v)
            Q_full[i, k:m] -= (2.0 * dot_iv / norm_v_sq) * v

    Q_red = Q_full[:, :n]
    R_red = np.triu(R[:n, :n])
    return Q_red, R_red


def compute_condition_number_l2(M: np.ndarray) -> float:
    """
    Calcula el número de condición espectral kappa_2(M) = sigma_max / sigma_min.
    """
    s = np.linalg.svd(M, compute_uv=False)
    if s[-1] <= 1e-16:
        return np.inf
    return float(s[0] / s[-1])


def solve_least_squares_normal_lu(A: np.ndarray, z: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Resuelve el problema de Mínimos Cuadrados mediante Ecuaciones Normales (A^T A) c = A^T z
    usando Factorización PA = LU con pivoteo parcial.
    """
    G = A.T @ A
    rhs = A.T @ z
    kappa_A = compute_condition_number_l2(A)
    kappa_G = compute_condition_number_l2(G)

    P, L, U = lu_partial_pivot(G)
    y = forward_substitution(L, P @ rhs)
    c = backward_substitution(U, y)

    residual_norm = float(np.linalg.norm(A @ c - z, ord=2))
    normal_eq_res = float(np.linalg.norm(G @ c - rhs, ord=2))

    metrics = {
        "kappa_A": kappa_A,
        "kappa_system": kappa_G,
        "residual_l2": residual_norm,
        "normal_eq_residual": normal_eq_res,
    }
    return c, metrics


def solve_least_squares_qr(A: np.ndarray, z: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Resuelve el problema de Mínimos Cuadrados directamente mediante Factorización QR
    de Householder (R c = Q^T z) sin formar A^T A.
    """
    Q, R = householder_qr(A)
    kappa_A = compute_condition_number_l2(A)
    kappa_R = compute_condition_number_l2(R)

    qty = Q.T @ z
    c = backward_substitution(R, qty)

    residual_norm = float(np.linalg.norm(A @ c - z, ord=2))
    orthogonality_error = float(np.linalg.norm(Q.T @ Q - np.eye(Q.shape[1]), ord="fro"))

    metrics = {
        "kappa_A": kappa_A,
        "kappa_system": kappa_R,
        "residual_l2": residual_norm,
        "orthogonality_error": orthogonality_error,
    }
    return c, metrics


## 3. Reconstrucción e Interpolación de Superficies sobre Datos Dispersos
Se implementan desde cero:
* `build_vandermonde_2d`: Matriz de diseño polinómica 2D para cualquier grado $d$.
* `BivariatePolynomialLS`: Mínimos Cuadrados Bivariados con soporte para resolver por QR o por LU.
* `PiecewiseLinearTriangularInterpolator`: Interpolación lineal continua sobre mallas triangulares vía coordenadas baricéntricas.
* `ModifiedShepardInterpolator` (MSHEP-QR): Método adaptativo de Cavoretto et al. (2024), con polinomios locales ajustados con nuestro QR de Householder y funciones de peso de soporte compacto.


In [ ]:
"""
Módulo: interpolators.py
Implementación desde cero de métodos de reconstrucción de superficies sobre datos dispersos:
1. Regresión Polinómica Bivariada por Mínimos Cuadrados (vía Factorización QR de Householder
   o vía Ecuaciones Normales con Factorización LU).
2. Interpolador Lineal por Partes sobre Elementos Triangulares (Coordenadas Baricéntricas).
3. Interpolador Adaptativo de Shepard Modificado (MSHEP, Cavoretto et al., 2024) con polinomios
   locales ajustados mediante QR de Householder.

Autores: Santiago Sandoval Cortés, Miguel David Moreno Lozano, Luis Alejandro Camacho Benítez
"""

    solve_least_squares_qr,
    solve_least_squares_normal_lu,
)


def build_vandermonde_2d(
    x: np.ndarray,
    y: np.ndarray,
    degree: int,
    scale_params: Optional[Dict[str, float]] = None,
) -> Tuple[np.ndarray, List[Tuple[int, int]]]:
    """
    Construye la matriz de diseño tipo Vandermonde 2D para polinomios de grado total <= degree.
    Orden de monomios: para total_deg = 0..degree, i = total_deg..0, j = total_deg - i.
    """
    x_arr = np.asarray(x, dtype=np.float64).ravel()
    y_arr = np.asarray(y, dtype=np.float64).ravel()

    if scale_params is not None:
        x_eval = (x_arr - scale_params["x_mid"]) / scale_params["x_half"]
        y_eval = (y_arr - scale_params["y_mid"]) / scale_params["y_half"]
    else:
        x_eval = x_arr
        y_eval = y_arr

    exponents: List[Tuple[int, int]] = []
    for d in range(degree + 1):
        for i in range(d, -1, -1):
            j = d - i
            exponents.append((i, j))

    n_pts = len(x_eval)
    n_terms = len(exponents)
    A = np.empty((n_pts, n_terms), dtype=np.float64)

    for col, (i, j) in enumerate(exponents):
        A[:, col] = (x_eval ** i) * (y_eval ** j)

    return A, exponents


class BivariatePolynomialLS:
    """
    Aproximador de superficie mediante Mínimos Cuadrados Polinómicos Bivariados.
    Permite seleccionar el método de resolución ('qr' o 'normal_lu') y activar/desactivar
    el escalamiento centrado de coordenadas para analizar el efecto del condicionamiento.
    """

    def __init__(self, degree: int = 2, method: str = "qr", normalize_coords: bool = True):
        self.degree = degree
        self.method = method.lower()
        self.normalize_coords = normalize_coords
        self.coeffs: Optional[np.ndarray] = None
        self.scale_params: Optional[Dict[str, float]] = None
        self.metrics: Dict[str, float] = {}

    def fit(self, x: np.ndarray, y: np.ndarray, z: np.ndarray) -> "BivariatePolynomialLS":
        x_arr = np.asarray(x, dtype=np.float64).ravel()
        y_arr = np.asarray(y, dtype=np.float64).ravel()
        z_arr = np.asarray(z, dtype=np.float64).ravel()

        if self.normalize_coords:
            x_min, x_max = float(np.min(x_arr)), float(np.max(x_arr))
            y_min, y_max = float(np.min(y_arr)), float(np.max(y_arr))
            self.scale_params = {
                "x_mid": 0.5 * (x_min + x_max),
                "x_half": max(0.5 * (x_max - x_min), 1e-12),
                "y_mid": 0.5 * (y_min + y_max),
                "y_half": max(0.5 * (y_max - y_min), 1e-12),
            }
        else:
            self.scale_params = None

        A, _ = build_vandermonde_2d(x_arr, y_arr, self.degree, self.scale_params)

        if self.method == "qr":
            self.coeffs, self.metrics = solve_least_squares_qr(A, z_arr)
        elif self.method == "normal_lu":
            self.coeffs, self.metrics = solve_least_squares_normal_lu(A, z_arr)
        else:
            raise ValueError(f"Método desconocido: {self.method}. Use 'qr' o 'normal_lu'.")

        return self

    def __call__(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        scalar_input = np.ndim(x) == 0 and np.ndim(y) == 0
        x_arr = np.atleast_1d(np.asarray(x, dtype=np.float64))
        y_arr = np.atleast_1d(np.asarray(y, dtype=np.float64))
        orig_shape = x_arr.shape

        A_eval, _ = build_vandermonde_2d(x_arr.ravel(), y_arr.ravel(), self.degree, self.scale_params)
        z_pred = A_eval @ self.coeffs

        if scalar_input:
            return float(z_pred[0])
        return z_pred.reshape(orig_shape)


class PiecewiseLinearTriangularInterpolator:
    """
    Interpolador lineal por partes sobre una triangulación de Delaunay implementando
    el cálculo directo de coordenadas baricéntricas en cada triángulo:
        Psi(x, y) = lambda_0 * z_0 + lambda_1 * z_1 + lambda_2 * z_2
    """

    def __init__(self, vertices: np.ndarray, triangles: np.ndarray, z_values: np.ndarray):
        self.vertices = np.asarray(vertices, dtype=np.float64)
        self.triangles = np.asarray(triangles, dtype=np.int64)
        self.z_values = np.asarray(z_values, dtype=np.float64).ravel()

        # Precomputar transformaciones baricéntricas para cada triángulo
        self.tri_v0 = self.vertices[self.triangles[:, 0]]
        self.tri_v1 = self.vertices[self.triangles[:, 1]]
        self.tri_v2 = self.vertices[self.triangles[:, 2]]
        self.tri_z = self.z_values[self.triangles]

    def _eval_single(self, px: float, py: float) -> float:
        # Encontrar el triángulo que contiene a (px, py) mediante coordenadas baricéntricas
        best_outside_dist = np.inf
        best_val = 0.0

        for idx in range(len(self.triangles)):
            x0, y0 = self.tri_v0[idx]
            x1, y1 = self.tri_v1[idx]
            x2, y2 = self.tri_v2[idx]

            detT = (y1 - y2) * (x0 - x2) + (x2 - x1) * (y0 - y2)
            if abs(detT) < 1e-15:
                continue

            l0 = ((y1 - y2) * (px - x2) + (x2 - x1) * (py - y2)) / detT
            l1 = ((y2 - y0) * (px - x2) + (x0 - x2) * (py - y2)) / detT
            l2 = 1.0 - l0 - l1

            min_l = min(l0, l1, l2)
            if min_l >= -1e-9:
                z0, z1, z2 = self.tri_z[idx]
                return float(l0 * z0 + l1 * z1 + l2 * z2)

            if -min_l < best_outside_dist:
                best_outside_dist = -min_l
                z0, z1, z2 = self.tri_z[idx]
                best_val = float(l0 * z0 + l1 * z1 + l2 * z2)

        return best_val

    def __call__(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        scalar_input = np.ndim(x) == 0 and np.ndim(y) == 0
        x_arr = np.atleast_1d(np.asarray(x, dtype=np.float64))
        y_arr = np.atleast_1d(np.asarray(y, dtype=np.float64))
        orig_shape = x_arr.shape

        x_flat = x_arr.ravel()
        y_flat = y_arr.ravel()
        out = np.empty(len(x_flat), dtype=np.float64)

        for i in range(len(x_flat)):
            out[i] = self._eval_single(float(x_flat[i]), float(y_flat[i]))

        if scalar_input:
            return float(out[0])
        return out.reshape(orig_shape)


class ModifiedShepardInterpolator:
    """
    Implementación desde cero del método de Shepard Modificado (MSHEP, Cavoretto et al., 2024):
    En cada nodo (x_i, y_i) se ajusta un polinomio local P_i(x, y) de grado 1 (lineal) por
    mínimos cuadrados ponderados sobre sus k_neighbors vecinos más cercanos, resuelto con
    nuestra Factorización QR de Householder. Luego se combinan mediante funciones de peso
    de soporte compacto de Franke-Little.
    """

    def __init__(self, k_neighbors: int = 8, local_degree: int = 1):
        self.k_neighbors = k_neighbors
        self.local_degree = local_degree
        self.nodes: Optional[np.ndarray] = None
        self.z_values: Optional[np.ndarray] = None
        self.local_coeffs: List[np.ndarray] = []
        self.radii_w: Optional[np.ndarray] = None

    def fit(self, x: np.ndarray, y: np.ndarray, z: np.ndarray) -> "ModifiedShepardInterpolator":
        x_arr = np.asarray(x, dtype=np.float64).ravel()
        y_arr = np.asarray(y, dtype=np.float64).ravel()
        z_arr = np.asarray(z, dtype=np.float64).ravel()
        n = len(x_arr)

        self.nodes = np.column_stack([x_arr, y_arr])
        self.z_values = z_arr
        self.local_coeffs = []
        self.radii_w = np.zeros(n, dtype=np.float64)

        k_eff = min(max(self.k_neighbors, 4), n - 1)

        for i in range(n):
            xi, yi, zi = x_arr[i], y_arr[i], z_arr[i]
            dists = np.sqrt((x_arr - xi) ** 2 + (y_arr - yi) ** 2)
            sorted_idx = np.argsort(dists)
            neigh_idx = sorted_idx[1 : k_eff + 1]
            r_max = float(dists[neigh_idx[-1]]) * 1.15
            self.radii_w[i] = max(r_max, 1e-9)

            # Ajustar P_i(x, y) = z_i + c1*(x - x_i) + c2*(y - y_i) [+ términos cuadráticos]
            dx = (x_arr[neigh_idx] - xi) / self.radii_w[i]
            dy = (y_arr[neigh_idx] - yi) / self.radii_w[i]
            dz = z_arr[neigh_idx] - zi

            d_neigh = dists[neigh_idx]
            omega = np.maximum(0.0, (self.radii_w[i] - d_neigh) / (self.radii_w[i] * (d_neigh + 1e-12)))

            if self.local_degree == 1 or len(neigh_idx) < 6:
                B = np.column_stack([dx, dy])
            else:
                B = np.column_stack([dx, dy, dx ** 2, dx * dy, dy ** 2])

            W_mat = B * omega[:, None]
            rhs_w = dz * omega
            try:
                c_loc, _ = solve_least_squares_qr(W_mat, rhs_w)
            except Exception:
                c_loc = np.zeros(B.shape[1], dtype=np.float64)

            self.local_coeffs.append(c_loc)

        return self

    def _eval_single(self, px: float, py: float) -> float:
        dists = np.sqrt((self.nodes[:, 0] - px) ** 2 + (self.nodes[:, 1] - py) ** 2)
        min_idx = int(np.argmin(dists))
        if dists[min_idx] < 1e-12:
            return float(self.z_values[min_idx])

        # Pesos de soporte compacto W_i(x, y) = [(R_i - d_i)_+ / (R_i * d_i)]^2
        raw_w = np.maximum(0.0, (self.radii_w - dists) / (self.radii_w * dists)) ** 2
        if np.sum(raw_w) <= 1e-15:
            # Respaldo de distancia inversa global si cae fuera de todos los radios locales
            raw_w = 1.0 / (dists ** 2)

        weights = raw_w / np.sum(raw_w)
        val = 0.0
        for i in range(len(self.z_values)):
            if weights[i] <= 0.0:
                continue
            xi, yi = self.nodes[i]
            dx = (px - xi) / self.radii_w[i]
            dy = (py - yi) / self.radii_w[i]
            c_loc = self.local_coeffs[i]
            if len(c_loc) == 2:
                pi_val = self.z_values[i] + c_loc[0] * dx + c_loc[1] * dy
            else:
                pi_val = (
                    self.z_values[i]
                    + c_loc[0] * dx
                    + c_loc[1] * dy
                    + c_loc[2] * (dx ** 2)
                    + c_loc[3] * (dx * dy)
                    + c_loc[4] * (dy ** 2)
                )
            val += weights[i] * pi_val

        return float(val)

    def __call__(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        scalar_input = np.ndim(x) == 0 and np.ndim(y) == 0
        x_arr = np.atleast_1d(np.asarray(x, dtype=np.float64))
        y_arr = np.atleast_1d(np.asarray(y, dtype=np.float64))
        orig_shape = x_arr.shape

        x_flat = x_arr.ravel()
        y_flat = y_arr.ravel()
        out = np.empty(len(x_flat), dtype=np.float64)

        for i in range(len(x_flat)):
            out[i] = self._eval_single(float(x_flat[i]), float(y_flat[i]))

        if scalar_input:
            return float(out[0])
        return out.reshape(orig_shape)


## 4. Cubatura Numérica, Refinamiento Adaptativo y Monte Carlo
Se implementan desde cero:
* `project_bogota_to_km`: Proyección métrica a coordenadas planas en kilómetros ($x, y$).
* `cubature_rule_pi_deg2` y `cubature_rule_pi_deg5`: Fórmulas con pesos positivos y nodos interiores (Reglas PI) de grado 2 (3 nodos) y grado 5 (7 nodos de Radon-Dunavant).
* `subdivide_triangle_1to4`: Bisección de aristas 1:4 en subtriángulos congruentes.
* `adaptive_cubature_mesh`: Algoritmo adaptativo con estimador encajado $|Q_5 - Q_2|$ y tolerancia ponderada por área.
* `uniform_cubature_mesh`: Cubatura sobre malla uniformemente refinada.
* `monte_carlo_cubature_mesh`: Simulación Monte Carlo sobre triángulos arbitrarios con semilla fija.
* `franke_function` y `exact_franke_integral_unit_square`: Benchmark de referencia analítica en $[0, 1]^2$.


In [ ]:
"""
Módulo: cubature_mesh.py
Implementación desde cero de:
1. Proyección geográfica a coordenadas métricas planas (km) para el dominio de Bogotá.
2. Fórmulas de cubatura con Pesos Positivos y Nodos Interiores (Reglas PI) de grado 2 (3 puntos)
   y grado 5 (7 puntos de Radon-Dunavant) sobre triángulos arbitrarios.
3. Algoritmo de Cubatura Adaptativa por bisección congruente 1:4 de aristas con indicador local
   de error encajado y tolerancia ponderada por área.
4. Cubatura sobre Malla Uniformemente Refinada y Simulación Monte Carlo en dominios triangulados
   con semilla pseudoaleatoria fija (seed=42).

Autores: Santiago Sandoval Cortés, Miguel David Moreno Lozano, Luis Alejandro Camacho Benítez
"""


EARTH_RADIUS_KM = 6371.0


def project_bogota_to_km(
    lon: np.ndarray,
    lat: np.ndarray,
    lon0: float = -74.10,
    lat0: float = 4.65,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Transforma coordenadas geográficas (Longitud, Latitud en grados) al plano métrico
    tangente local centrado en Bogotá (lon0, lat0) expresado en kilómetros (km).
    
    x = R_earth * cos(lat0) * (lon - lon0) * pi / 180   [km Este]
    y = R_earth * (lat - lat0) * pi / 180               [km Norte]
    """
    lon_arr = np.asarray(lon, dtype=np.float64)
    lat_arr = np.asarray(lat, dtype=np.float64)
    deg2rad = np.pi / 180.0
    cos_lat0 = np.cos(lat0 * deg2rad)

    x_km = EARTH_RADIUS_KM * cos_lat0 * (lon_arr - lon0) * deg2rad
    y_km = EARTH_RADIUS_KM * (lat_arr - lat0) * deg2rad
    return x_km, y_km


def triangle_area(v0: np.ndarray, v1: np.ndarray, v2: np.ndarray) -> float:
    """
    Calcula el área exacta de un triángulo plano con vértices v0, v1, v2 mediante el
    determinante jacobiano: Area = 0.5 * |(x1 - x0)*(y2 - y0) - (x2 - x0)*(y1 - y0)|.
    """
    det = (v1[0] - v0[0]) * (v2[1] - v0[1]) - (v2[0] - v0[0]) * (v1[1] - v0[1])
    return 0.5 * abs(float(det))


def cubature_rule_pi_deg2() -> Tuple[np.ndarray, np.ndarray]:
    """
    Regla de cubatura PI (Pesos Positivos y Nodos Interiores) de grado algebraico 2
    con 3 nodos interiores en coordenadas baricéntricas (lambda_0, lambda_1, lambda_2).
    La suma de pesos es 1.0 (multiplicada por el área del triángulo).
    """
    bary = np.array(
        [
            [2.0 / 3.0, 1.0 / 6.0, 1.0 / 6.0],
            [1.0 / 6.0, 2.0 / 3.0, 1.0 / 6.0],
            [1.0 / 6.0, 1.0 / 6.0, 2.0 / 3.0],
        ],
        dtype=np.float64,
    )
    weights = np.array([1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0], dtype=np.float64)
    return bary, weights


def cubature_rule_pi_deg5() -> Tuple[np.ndarray, np.ndarray]:
    """
    Regla de cubatura PI (Pesos Positivos y Nodos Interiores) de Radon-Dunavant de
    grado algebraico 5 con 7 nodos estrictamente interiores y pesos estrictamente positivos.
    """
    sqrt15 = np.sqrt(15.0)
    a1 = (6.0 - sqrt15) / 21.0
    b1 = (9.0 + 2.0 * sqrt15) / 21.0
    w1 = (155.0 - sqrt15) / 1200.0

    a2 = (6.0 + sqrt15) / 21.0
    b2 = (9.0 - 2.0 * sqrt15) / 21.0
    w2 = (155.0 + sqrt15) / 1200.0

    w0 = 9.0 / 40.0

    bary = np.array(
        [
            [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0],
            [b1, a1, a1],
            [a1, b1, a1],
            [a1, a1, b1],
            [b2, a2, a2],
            [a2, b2, a2],
            [a2, a2, b2],
        ],
        dtype=np.float64,
    )
    weights = np.array([w0, w1, w1, w1, w2, w2, w2], dtype=np.float64)
    return bary, weights


BARY_DEG2, WEIGHTS_DEG2 = cubature_rule_pi_deg2()
BARY_DEG5, WEIGHTS_DEG5 = cubature_rule_pi_deg5()


def integrate_single_triangle(
    func: Callable[[np.ndarray, np.ndarray], np.ndarray],
    v0: np.ndarray,
    v1: np.ndarray,
    v2: np.ndarray,
    bary: np.ndarray,
    weights: np.ndarray,
) -> Tuple[float, int]:
    """
    Evalúa la regla de cubatura sobre un triángulo físico con vértices (v0, v1, v2).
    Retorna (valor_integral, numero_evaluaciones).
    """
    area = triangle_area(v0, v1, v2)
    if area <= 1e-16:
        return 0.0, 0

    pts_x = bary[:, 0] * v0[0] + bary[:, 1] * v1[0] + bary[:, 2] * v2[0]
    pts_y = bary[:, 0] * v0[1] + bary[:, 1] * v1[1] + bary[:, 2] * v2[1]

    f_vals = np.asarray(func(pts_x, pts_y), dtype=np.float64).ravel()
    integral = area * float(np.dot(weights, f_vals))
    return integral, len(weights)


def subdivide_triangle_1to4(
    v0: np.ndarray, v1: np.ndarray, v2: np.ndarray
) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Subdivide un triángulo (v0, v1, v2) en 4 subtriángulos congruentes conectando
    los puntos medios de sus tres aristas (refinamiento 1:4 de aristas).
    """
    m01 = 0.5 * (v0 + v1)
    m12 = 0.5 * (v1 + v2)
    m20 = 0.5 * (v2 + v0)
    return [
        (v0, m01, m20),
        (m01, v1, m12),
        (m20, m12, v2),
        (m01, m12, m20),
    ]


def adaptive_cubature_mesh(
    func: Callable[[np.ndarray, np.ndarray], np.ndarray],
    vertices: np.ndarray,
    triangles: np.ndarray,
    tol_abs: float = 1e-4,
    max_depth: int = 5,
    min_area_ratio: float = 1e-4,
) -> Dict:
    """
    Ejecuta el algoritmo de Cubatura Adaptativa sobre la malla triangular inicial.
    Criterio de refinamiento en cada celda T_e:
        |Q_5(T_e) - Q_2(T_e)| > tol_abs * (Area(T_e) / Area(Omega))
        y depth < max_depth y Area(T_e) > min_area.
    """
    t0 = time.perf_counter()
    verts = np.asarray(vertices, dtype=np.float64)
    tris = np.asarray(triangles, dtype=np.int64)

    initial_cells = []
    total_area = 0.0
    for tri in tris:
        v0, v1, v2 = verts[tri[0]], verts[tri[1]], verts[tri[2]]
        a = triangle_area(v0, v1, v2)
        total_area += a
        initial_cells.append((v0, v1, v2, 0))

    min_area = total_area * min_area_ratio
    total_integral = 0.0
    total_error_est = 0.0
    total_evals = 0
    final_triangles: List[Tuple[np.ndarray, np.ndarray, np.ndarray, int, float]] = []

    stack = list(reversed(initial_cells))
    while stack:
        v0, v1, v2, depth = stack.pop()
        cell_area = triangle_area(v0, v1, v2)

        q2, ev2 = integrate_single_triangle(func, v0, v1, v2, BARY_DEG2, WEIGHTS_DEG2)
        q5, ev5 = integrate_single_triangle(func, v0, v1, v2, BARY_DEG5, WEIGHTS_DEG5)
        total_evals += ev2 + ev5

        err_local = abs(q5 - q2)
        tol_local = tol_abs * (cell_area / total_area)

        if (err_local > tol_local) and (depth < max_depth) and (cell_area > min_area):
            children = subdivide_triangle_1to4(v0, v1, v2)
            for ch_v0, ch_v1, ch_v2 in reversed(children):
                stack.append((ch_v0, ch_v1, ch_v2, depth + 1))
        else:
            total_integral += q5
            total_error_est += err_local
            final_triangles.append((v0, v1, v2, depth, err_local))

    elapsed = time.perf_counter() - t0
    return {
        "integral": total_integral,
        "area_total": total_area,
        "mean_value": total_integral / total_area,
        "error_estimate": total_error_est,
        "n_evaluations": total_evals,
        "n_Initial_triangles": len(initial_cells),
        "n_final_triangles": len(final_triangles),
        "final_triangles": final_triangles,
        "time_sec": elapsed,
    }


def uniform_cubature_mesh(
    func: Callable[[np.ndarray, np.ndarray], np.ndarray],
    vertices: np.ndarray,
    triangles: np.ndarray,
    refinement_Check_depth: int = 0,
    rule_degree: int = 5,
) -> Dict:
    """
    Evalúa la cubatura sobre una malla uniformemente subdividida hasta profundidad
    `refinement_Check_depth` (cada nivel multiplica por 4 el número de triángulos).
    """
    t0 = time.perf_counter()
    verts = np.asarray(vertices, dtype=np.float64)
    tris = np.asarray(triangles, dtype=np.int64)

    cells = [(verts[t[0]], verts[t[1]], verts[t[2]]) for t in tris]
    for _ in range(refinement_Check_depth):
        new_cells = []
        for v0, v1, v2 in cells:
            new_cells.extend(subdivide_triangle_1to4(v0, v1, v2))
        cells = new_cells

    bary, weights = (BARY_DEG5, WEIGHTS_DEG5) if rule_degree == 5 else (BARY_DEG2, WEIGHTS_DEG2)
    total_integral = 0.0
    total_area = 0.0
    total_evals = 0

    for v0, v1, v2 in cells:
        total_area += triangle_area(v0, v1, v2)
        q, ev = integrate_single_triangle(func, v0, v1, v2, bary, weights)
        total_integral += q
        total_evals += ev

    elapsed = time.perf_counter() - t0
    return {
        "integral": total_integral,
        "area_total": total_area,
        "mean_value": total_integral / total_area,
        "n_evaluations": total_evals,
        "n_triangles": len(cells),
        "time_sec": elapsed,
    }


def monte_carlo_cubature_mesh(
    func: Callable[[np.ndarray, np.ndarray], np.ndarray],
    vertices: np.ndarray,
    triangles: np.ndarray,
    n_samples: int = 10000,
    seed: int = 42,
) -> Dict:
    """
    Integración numérica por Simulación Monte Carlo sobre el dominio poligonal
    definido por la malla de triángulos, con semilla pseudoaleatoria fija (seed=42).
    Cada triángulo se muestrea proporcionalmente a su área exacta y los puntos
    interiores se generan uniformemente mediante transformación baricéntrica de raíz cuadrada.
    """
    t0 = time.perf_counter()
    rng = np.random.default_rng(seed)

    verts = np.asarray(vertices, dtype=np.float64)
    tris = np.asarray(triangles, dtype=np.int64)

    areas = np.array([triangle_area(verts[t[0]], verts[t[1]], verts[t[2]]) for t in tris], dtype=np.float64)
    total_area = float(np.sum(areas))
    probs = areas / total_area

    chosen_tris = rng.choice(len(tris), size=n_samples, p=probs)
    u1 = rng.uniform(0.0, 1.0, size=n_samples)
    u2 = rng.uniform(0.0, 1.0, size=n_samples)

    sqrt_u1 = np.sqrt(u1)
    l0 = 1.0 - sqrt_u1
    l1 = sqrt_u1 * (1.0 - u2)
    l2 = sqrt_u1 * u2

    v0_all = verts[tris[chosen_tris, 0]]
    v1_all = verts[tris[chosen_tris, 1]]
    v2_all = verts[tris[chosen_tris, 2]]

    px = l0 * v0_all[:, 0] + l1 * v1_all[:, 0] + l2 * v2_all[:, 0]
    py = l0 * v0_all[:, 1] + l1 * v1_all[:, 1] + l2 * v2_all[:, 1]

    f_vals = np.asarray(func(px, py), dtype=np.float64).ravel()
    mean_f = float(np.mean(f_vals))
    std_f = float(np.std(f_vals, ddof=1))

    integral = total_area * mean_f
    std_error = total_area * (std_f / np.sqrt(n_samples))
    elapsed = time.perf_counter() - t0

    return {
        "integral": integral,
        "area_total": total_area,
        "mean_value": mean_f,
        "std_error": std_error,
        "n_evaluations": n_samples,
        "time_sec": elapsed,
    }


def franke_function(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Función analítica estándar de prueba de Franke (1979) sobre [0, 1]^2.
    """
    t1 = 0.75 * np.exp(-((9.0 * x - 2.0) ** 2 + (9.0 * y - 2.0) ** 2) / 4.0)
    t2 = 0.75 * np.exp(-((9.0 * x + 1.0) ** 2) / 49.0 - (9.0 * y + 1.0) / 10.0)
    t3 = 0.50 * np.exp(-((9.0 * x - 7.0) ** 2 + (9.0 * y - 3.0) ** 2) / 4.0)
    t4 = -0.20 * np.exp(-((9.0 * x - 4.0) ** 2 + (9.0 * y - 7.0) ** 2))
    return t1 + t2 + t3 + t4


def exact_franke_integral_unit_square() -> float:
    """
    Calcula el valor exacto de referencia (Ground Truth a precisión de máquina IEEE-754)
    de la integral de Franke en [0, 1]^2 mediante producto tensorial de Gauss-Legendre de 64 nodos.
    """
    nodes_1d, weights_1d = np.polynomial.legendre.leggauss(64)
    # Transformar de [-1, 1] a [0, 1]
    x_1d = 0.5 * (nodes_1d + 1.0)
    w_1d = 0.5 * weights_1d

    X, Y = np.meshgrid(x_1d, x_1d)
    W = np.outer(w_1d, w_1d)
    F = franke_function(X, Y)
    return float(np.sum(W * F))


## 5. Experimento 1: Estabilidad Numérica (QR de Householder vs. Ecuaciones Normales LU)
Compara el número de condición espectral $\kappa_2$ y el error relativo en los coeficientes al aumentar el grado polinómico $d = 1, 2, 3, 4$.


In [ ]:
print("=== Experimento 1: Comparación de Estabilidad QR vs. LU ===")
rng = np.random.default_rng(SEED)
n_pts = 65
x_sim = 10.0 + rng.uniform(0.0, 1.0, n_pts)
y_sim = 10.0 + rng.uniform(0.0, 1.0, n_pts)
z_sim = franke_function(x_sim - 10.0, y_sim - 10.0)

exp1_records = []
for deg in [1, 2, 3, 4]:
    A_raw, _ = build_vandermonde_2d(x_sim, y_sim, degree=deg, scale_params=None)
    c_ref, _, _, _ = np.linalg.lstsq(A_raw, z_sim, rcond=None)

    # 1. QR Householder (propio)
    t0 = time.perf_counter()
    c_qr, m_qr = solve_least_squares_qr(A_raw, z_sim)
    t_qr = time.perf_counter() - t0
    err_rel_qr = float(np.linalg.norm(c_qr - c_ref) / (np.linalg.norm(c_ref) + 1e-16))

    # 2. LU sobre Ecuaciones Normales (propio)
    t0 = time.perf_counter()
    try:
        c_lu, m_lu = solve_least_squares_normal_lu(A_raw, z_sim)
        t_lu = time.perf_counter() - t0
        err_rel_lu = float(np.linalg.norm(c_lu - c_ref) / (np.linalg.norm(c_ref) + 1e-16))
        kappa_ata = m_lu["kappa_system"]
    except Exception:
        t_lu = np.nan
        err_rel_lu = 1.0
        kappa_ata = m_qr["kappa_A"] ** 2

    exp1_records.append({
        "Grado (d)": deg,
        "Términos (p)": A_raw.shape[1],
        "kappa_2(R) (QR)": m_qr["kappa_system"],
        "kappa_2(A^T A) (LU)": kappa_ata,
        "Error Rel. QR": err_rel_qr,
        "Error Rel. LU": err_rel_lu,
        "Ortogonalidad ||Q^T Q - I||_F": m_qr["orthogonality_error"],
        "Tiempo QR (ms)": t_qr * 1000.0,
        "Tiempo LU (ms)": t_lu * 1000.0
    })

df_exp1 = pd.DataFrame(exp1_records)
display(df_exp1[["Grado (d)", "Términos (p)", "kappa_2(R) (QR)", "kappa_2(A^T A) (LU)", "Error Rel. QR", "Error Rel. LU"]])

# Gráfica Figura 1
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.2), dpi=150)
degs = df_exp1["Grado (d)"].values
ax1.semilogy(degs, df_exp1["kappa_2(R) (QR)"], "o-", color="#0A2540", lw=2.2, markersize=7, label=r"$\kappa_2(R) = \kappa_2(A)$ (QR Householder)")
ax1.semilogy(degs, df_exp1["kappa_2(A^T A) (LU)"], "s--", color="#C53030", lw=2.2, markersize=7, label=r"$\kappa_2(A^T A) = \kappa_2(A)^2$ (Ec. Normales LU)")
ax1.axhline(1.0 / np.finfo(float).eps, color="#718096", ls=":", lw=1.2, label=r"Límite IEEE-754 ($\epsilon_{\mathrm{mach}}^{-1} pprox 4.5	imes 10^{15}$)")
ax1.set_title("A. Crecimiento del Número de Condición", fontweight="bold", color="#0A2540")
ax1.set_xlabel("Grado Polinómico Total ($d$)", fontweight="bold")
ax1.set_ylabel(r"Número de Condición Espectral ($\log_{10}$)", fontweight="bold")
ax1.set_xticks(degs)
ax1.legend(fontsize=8.5, frameon=True, facecolor="white")
ax1.grid(True, which="both", ls="--", alpha=0.45)

ax2.semilogy(degs, df_exp1["Error Rel. QR"], "o-", color="#2B6CB0", lw=2.2, markersize=7, label="Error Relativo con QR (Householder)")
ax2.semilogy(degs, df_exp1["Error Rel. LU"], "s--", color="#DD6B20", lw=2.2, markersize=7, label="Error Relativo con LU (Ec. Normales)")
ax2.set_title("B. Pérdida de Precisión en Coeficientes", fontweight="bold", color="#0A2540")
ax2.set_xlabel("Grado Polinómico Total ($d$)", fontweight="bold")
ax2.set_ylabel("Error Relativo en Coeficientes", fontweight="bold")
ax2.set_xticks(degs)
ax2.legend(fontsize=8.5, frameon=True, facecolor="white")
ax2.grid(True, which="both", ls="--", alpha=0.45)
plt.tight_layout()
plt.show()


## 6. Experimento 2: Benchmark de Franke (Convergencia y Error Stalling)
Evalúa la tasa de convergencia de la cubatura adaptativa frente a malla uniforme y Monte Carlo, y comprueba el estancamiento del error (*Error Stalling*) al integrar desde nodos dispersos finitos.


In [ ]:
print("=== Experimento 2: Benchmark de Franke ===")
I_exact = exact_franke_integral_unit_square()
print(f"Integral exacta de Franke en [0,1]^2: I_exact = {I_exact:.12f}")

verts_sq = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]], dtype=np.float64)
tris_sq = np.array([[0, 1, 2], [0, 2, 3]], dtype=np.int64)

conv_rows = []
# Malla Uniforme
for depth in [0, 1, 2, 3, 4]:
    res_u = uniform_cubature_mesh(franke_function, verts_sq, tris_sq, refinement_Check_depth=depth, rule_degree=5)
    err_rel = abs(res_u["integral"] - I_exact) / abs(I_exact)
    conv_rows.append({
        "Método": "Cubatura Uniforme (PI Grado 5)",
        "Parámetro": f"depth={depth}",
        "Evaluaciones (M)": res_u["n_evaluations"],
        "Valor Integral": res_u["integral"],
        "Error Absoluto": abs(res_u["integral"] - I_exact),
        "Error Relativo": err_rel,
        "Tiempo (ms)": res_u["time_sec"] * 1000.0
    })

# Cubatura Adaptativa
for tol in [2e-2, 5e-3, 8e-4, 1.5e-4, 2e-5]:
    res_a = adaptive_cubature_mesh(franke_function, verts_sq, tris_sq, tol_abs=tol, max_depth=6, min_area_ratio=1e-5)
    err_rel = abs(res_a["integral"] - I_exact) / abs(I_exact)
    conv_rows.append({
        "Método": "Cubatura Adaptativa (Directa)",
        "Parámetro": f"tol={tol:.1e}",
        "Evaluaciones (M)": res_a["n_final_triangles"] * 7,
        "Valor Integral": res_a["integral"],
        "Error Absoluto": abs(res_a["integral"] - I_exact),
        "Error Relativo": err_rel,
        "Tiempo (ms)": res_a["time_sec"] * 1000.0
    })

# Simulación Monte Carlo
for k_mc in [50, 200, 1000, 5000, 25000]:
    res_mc = monte_carlo_cubature_mesh(franke_function, verts_sq, tris_sq, n_samples=k_mc, seed=SEED)
    err_rel = abs(res_mc["integral"] - I_exact) / abs(I_exact)
    conv_rows.append({
        "Método": "Simulación Monte Carlo (seed=42)",
        "Parámetro": f"K={k_mc}",
        "Evaluaciones (M)": res_mc["n_evaluations"],
        "Valor Integral": res_mc["integral"],
        "Error Absoluto": abs(res_mc["integral"] - I_exact),
        "Error Relativo": err_rel,
        "Tiempo (ms)": res_mc["time_sec"] * 1000.0
    })

# Error Stalling con datos dispersos
for n_scattered in [35, 250]:
    xs = rng.uniform(0.0, 1.0, n_scattered)
    ys = rng.uniform(0.0, 1.0, n_scattered)
    zs = franke_function(xs, ys)
    mshep = ModifiedShepardInterpolator(k_neighbors=11, local_degree=2).fit(xs, ys, zs)

    for depth in [0, 1, 2, 3, 4]:
        res_st = uniform_cubature_mesh(mshep, verts_sq, tris_sq, refinement_Check_depth=depth, rule_degree=5)
        err_rel = abs(res_st["integral"] - I_exact) / abs(I_exact)
        conv_rows.append({
            "Método": f"Cubatura sobre MSHEP-QR (N={n_scattered})",
            "Parámetro": f"depth={depth}",
            "Evaluaciones (M)": res_st["n_evaluations"],
            "Valor Integral": res_st["integral"],
            "Error Absoluto": abs(res_st["integral"] - I_exact),
            "Error Relativo": err_rel,
            "Tiempo (ms)": res_st["time_sec"] * 1000.0
        })

df_conv = pd.DataFrame(conv_rows)
display(df_conv.head(11))

# Gráfica Figura 2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.3), dpi=150)
sub_adap = df_conv[df_conv["Método"] == "Cubatura Adaptativa (Directa)"]
sub_unif = df_conv[df_conv["Método"] == "Cubatura Uniforme (PI Grado 5)"]
sub_mc = df_conv[df_conv["Método"] == "Simulación Monte Carlo (seed=42)"]

ax1.loglog(sub_adap["Evaluaciones (M)"], sub_adap["Error Relativo"], "o-", color="#0A2540", lw=2.2, markersize=6, label="Cubatura Adaptativa (PI 3/7 pts)")
ax1.loglog(sub_unif["Evaluaciones (M)"], sub_unif["Error Relativo"], "s--", color="#2B6CB0", lw=2.0, markersize=6, label="Cubatura Malla Uniforme")
ax1.loglog(sub_mc["Evaluaciones (M)"], sub_mc["Error Relativo"], "^:", color="#C53030", lw=2.0, markersize=6, label="Monte Carlo (O(K^-1/2))")
ax1.set_title("A. Convergencia: Error Relativo vs Evaluaciones", fontweight="bold", color="#0A2540")
ax1.set_xlabel("Número de Evaluaciones de Función (M)", fontweight="bold")
ax1.set_ylabel("Error Relativo |I - Q| / |I|", fontweight="bold")
ax1.legend(fontsize=8.5, frameon=True, facecolor="white")
ax1.grid(True, which="both", ls="--", alpha=0.45)

sub_s35 = df_conv[df_conv["Método"] == "Cubatura sobre MSHEP-QR (N=35)"]
sub_s250 = df_conv[df_conv["Método"] == "Cubatura sobre MSHEP-QR (N=250)"]

ax2.loglog(sub_unif["Evaluaciones (M)"], sub_unif["Error Relativo"], "k--", lw=1.6, alpha=0.7, label="Sin error de interpolación (f exacta)")
ax2.loglog(sub_s35["Evaluaciones (M)"], sub_s35["Error Relativo"], "d-", color="#DD6B20", lw=2.2, markersize=6, label="Datos Dispersos N=35 (MSHEP-QR)")
ax2.loglog(sub_s250["Evaluaciones (M)"], sub_s250["Error Relativo"], "o-", color="#2F855A", lw=2.2, markersize=6, label="Datos Dispersos N=250 (MSHEP-QR)")
ax2.set_title("B. Verificación de Error Stalling", fontweight="bold", color="#0A2540")
ax2.set_xlabel("Nodos de Cubatura Evaluados (M)", fontweight="bold")
ax2.set_ylabel("Error Relativo Total", fontweight="bold")
ax2.legend(fontsize=8.5, frameon=True, facecolor="white")
ax2.grid(True, which="both", ls="--", alpha=0.45)
plt.tight_layout()
plt.show()


## 7. Experimento 3: Aplicación a la Calidad del Aire en Bogotá ($PM_{10}$, RMCAB 2023)
Integra las 20 estaciones de Bogotá sobre su dominio métrico plano ($	ext{km}^2$), calcula la Concentración Media Espacial $\overline{C}_\Omega$ y la Masa Atmosférica Acumulada en la capa de mezcla urbana.


In [ ]:
print("=== Experimento 3: Calidad del Aire en Bogotá ===")
csv_path = "data/calidad_aire_bogota_pm10_2023.csv"
if not os.path.exists(csv_path):
    csv_path = "Segunda_Entrega/data/calidad_aire_bogota_pm10_2023.csv"

df_bog = pd.read_csv(csv_path)
x_km, y_km = project_bogota_to_km(df_bog["longitud"].values, df_bog["latitud"].values)
df_bog["x_km"] = x_km
df_bog["y_km"] = y_km
z_pm10 = df_bog["promedio"].values

verts_bog = np.column_stack([x_km, y_km])
triang = mtri.Triangulation(x_km, y_km)
tris_bog = triang.triangles

# Ajustar modelos
poly_qr = BivariatePolynomialLS(degree=2, method="qr", normalize_coords=True).fit(x_km, y_km, z_pm10)
poly_lu = BivariatePolynomialLS(degree=2, method="normal_lu", normalize_coords=True).fit(x_km, y_km, z_pm10)
mshep_qr = ModifiedShepardInterpolator(k_neighbors=7, local_degree=1).fit(x_km, y_km, z_pm10)
lin_fem = PiecewiseLinearTriangularInterpolator(verts_bog, tris_bog, z_pm10)

# Cubaturas
res_poly_qr = adaptive_cubature_mesh(poly_qr, verts_bog, tris_bog, tol_abs=5.0, max_depth=4, min_area_ratio=1e-3)
res_poly_lu = adaptive_cubature_mesh(poly_lu, verts_bog, tris_bog, tol_abs=5.0, max_depth=4, min_area_ratio=1e-3)
res_mshep_adap = adaptive_cubature_mesh(mshep_qr, verts_bog, tris_bog, tol_abs=15.0, max_depth=4, min_area_ratio=2e-3)
res_mshep_unif = uniform_cubature_mesh(mshep_qr, verts_bog, tris_bog, refinement_Check_depth=2, rule_degree=5)
res_lin_fem = adaptive_cubature_mesh(lin_fem, verts_bog, tris_bog, tol_abs=5.0, max_depth=3, min_area_ratio=1e-3)
res_mc_mshep = monte_carlo_cubature_mesh(mshep_qr, verts_bog, tris_bog, n_samples=10000, seed=SEED)

methods_list = [
    ("Polinomio Mín. Cuadrados (d=2, QR) + Adaptativa", res_poly_qr),
    ("Polinomio Mín. Cuadrados (d=2, LU) + Adaptativa", res_poly_lu),
    ("Shepard Modificado (MSHEP-QR) + Adaptativa", res_mshep_adap),
    ("Shepard Modificado (MSHEP-QR) + Uniforme (k=2)", res_mshep_unif),
    ("Interpolación Lineal Baricéntrica + Adaptativa", res_lin_fem),
    ("Simulación Monte Carlo (K=10,000) sobre MSHEP-QR", res_mc_mshep),
]

bog_summary = []
for name, res in methods_list:
    I_val = res["integral"]
    area_km2 = res["area_total"]
    c_mean = res["mean_value"]
    mass_500m_kg = I_val * 0.5
    mass_1000m_kg = I_val * 1.0
    bog_summary.append({
        "Método": name,
        "Área (km2)": area_km2,
        "Integral I_Omega (g/m)": I_val,
        "Conc. Media Espacial (ug/m3)": c_mean,
        "Masa Capa 500m (kg)": mass_500m_kg,
        "Masa Capa 1000m (kg)": mass_1000m_kg,
        "Evaluaciones (M)": res["n_evaluations"],
        "Tiempo (ms)": res["time_sec"] * 1000.0
    })

df_bog_table = pd.DataFrame(bog_summary)
display(df_bog_table[["Método", "Conc. Media Espacial (ug/m3)", "Masa Capa 500m (kg)", "Masa Capa 1000m (kg)", "Evaluaciones (M)", "Tiempo (ms)"]])

# Gráfica Figura 3: Malla
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.8), dpi=150, sharex=True, sharey=True)
tpc1 = ax1.tripcolor(triang, z_pm10, cmap="YlOrRd", shading="gouraud", alpha=0.82)
ax1.triplot(triang, color="#1A365D", lw=1.1, ls="--", alpha=0.85)
ax1.scatter(x_km, y_km, c="black", s=38, zorder=5, label="Estaciones RMCAB (20)")
ax1.set_title(f"A. Malla Inicial ({len(tris_bog)} Celdas, Área={res_mshep_adap['area_total']:.1f} km2)", fontweight="bold", color="#0A2540")
ax1.set_xlabel("Coordenada Este x (km)", fontweight="bold")
ax1.set_ylabel("Coordenada Norte y (km)", fontweight="bold")
ax1.legend(loc="upper left", frameon=True, facecolor="white", fontsize=8.5)

tpc2 = ax2.tripcolor(triang, z_pm10, cmap="YlOrRd", shading="gouraud", alpha=0.82)
for v0, v1, v2, depth, _ in res_mshep_adap["final_triangles"]:
    poly_x = [v0[0], v1[0], v2[0], v0[0]]
    poly_y = [v0[1], v1[1], v2[1], v0[1]]
    col = "#0A2540" if depth >= 2 else "#2B6CB0"
    ax2.plot(poly_x, poly_y, color=col, lw=0.65 if depth > 0 else 1.0, alpha=0.88)
ax2.scatter(x_km, y_km, c="black", s=35, zorder=5)
ax2.set_title(f"B. Malla Adaptativa Refinada ({res_mshep_adap['n_final_triangles']} Celdas)", fontweight="bold", color="#0A2540")
ax2.set_xlabel("Coordenada Este x (km)", fontweight="bold")
cbar = fig.colorbar(tpc2, ax=[ax1, ax2], shrink=0.86, pad=0.02)
cbar.set_label("Concentración PM10 (ug/m3)", fontweight="bold")
plt.show()

# Gráfica Figura 4: Comparación
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.1), dpi=150)
labels = ["Polinomio QR", "Polinomio LU", "Shepard MSHEP", "Lineal Baric.", "Monte Carlo"]
idx = [0, 1, 2, 4, 5]
c_vals = [bog_summary[i]["Conc. Media Espacial (ug/m3)"] for i in idx]
m_vals = [bog_summary[i]["Evaluaciones (M)"] for i in idx]
arith_mean = float(np.mean(z_pm10))

colors = ["#0A2540", "#4A5568", "#2B6CB0", "#319795", "#C53030"]
ax1.bar(labels, c_vals, color=colors, edgecolor="black", width=0.55, alpha=0.9)
ax1.axhline(arith_mean, color="#D69E2E", ls="--", lw=2.0, label=f"Media Aritmética Simple ({arith_mean:.2f} ug/m3)")
ax1.set_ylim(0, 42)
ax1.set_title("A. Concentración Media Espacial Ponderada", fontweight="bold", color="#0A2540")
ax1.set_ylabel("Concentración PM10 (ug/m3)", fontweight="bold")
ax1.legend(loc="upper right", frameon=True, facecolor="white", fontsize=8.2)

ax2.bar(labels, m_vals, color=colors, edgecolor="black", width=0.55, alpha=0.9)
ax2.set_yscale("log")
ax2.set_title("B. Costo Computacional (Evaluaciones)", fontweight="bold", color="#0A2540")
ax2.set_ylabel("Evaluaciones M (Escala Log)", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================================
# VALIDACIÓN FORMAL CONTRA LIBRERÍA DE REFERENCIA (NUMERAL 1.4 Y 8.3 DE LA GUÍA)
# ==========================================================================
# Comparamos nuestro interpolador baricéntrico por triángulos implementado desde cero
# contra la implementación estándar de referencia: matplotlib.tri.LinearTriInterpolator
ref_interp = mtri.LinearTriInterpolator(triang, z_pm10)

# Evaluar en 500 puntos aleatorios uniformemente distribuidos en el interior de los triángulos
rng = np.random.default_rng(42)
rand_tris = rng.integers(0, len(tris_bog), size=500)
r1 = rng.uniform(0, 1, size=500)
r2 = rng.uniform(0, 1, size=500)
mask = r1 + r2 > 1.0
r1[mask] = 1.0 - r1[mask]
r2[mask] = 1.0 - r2[mask]

px = verts_bog[tris_bog[rand_tris, 0], 0] + r1 * (verts_bog[tris_bog[rand_tris, 1], 0] - verts_bog[tris_bog[rand_tris, 0], 0]) + r2 * (verts_bog[tris_bog[rand_tris, 2], 0] - verts_bog[tris_bog[rand_tris, 0], 0])
py = verts_bog[tris_bog[rand_tris, 0], 1] + r1 * (verts_bog[tris_bog[rand_tris, 1], 1] - verts_bog[tris_bog[rand_tris, 0], 1]) + r2 * (verts_bog[tris_bog[rand_tris, 2], 1] - verts_bog[tris_bog[rand_tris, 0], 1])

val_propio = lin_fem(px, py)
val_referencia = ref_interp(px, py)
max_discrepancia = np.max(np.abs(val_propio - val_referencia))

print(f'>> DISCREPANCIA MÁXIMA FRENTE A LIBRERÍA DE REFERENCIA: {max_discrepancia:.2e}')
if max_discrepancia < 1e-11:
    print('>> [VALIDACIÓN EXITOSA]: El código propio coincide exactamente a nivel de precisión de máquina.')


In [ ]:
# ==========================================================================
# ANÁLISIS DE SENSIBILIDAD DE HIPERPARÁMETROS EN SHEPARD MODIFICADO (k_neighbors)
# ==========================================================================
sens_records = []
for k_val in [5, 6, 7, 8, 10]:
    m_test = ModifiedShepardInterpolator(k_neighbors=k_val, local_degree=1).fit(x_km, y_km, z_pm10)
    r_test = adaptive_cubature_mesh(m_test, verts_bog, tris_bog, tol_abs=15.0, max_depth=4, min_area_ratio=2e-3)
    sens_records.append({
        'k_vecinos': k_val,
        'Concentración_Media (ug/m3)': round(r_test['mean_value'], 2),
        'Integral_Superficie (g/m)': round(r_test['integral'], 1),
        'Masa_500m (Ton)': round(r_test['integral'] * 0.5 / 1000.0, 2),
        'Celdas_Finales': r_test['n_final_triangles']
    })
df_sensibilidad = pd.DataFrame(sens_records)
print('--- Sensibilidad de la Cubatura frente al número de vecinos k en Shepard ---')
display(df_sensibilidad)
